# Grover mixer compared to ordinary QAOA

All 15 source stages, four qubits and three layers in both the ordinary and Grover-mixer circuits, two budgets of 30 objective queries with 1000 actual draws per query, two independently sampled 1000-draw final distributions, and side-by-side comparison.

Use native COBYLA with the source linear pi schedule and a hard cap of 30 actual objective queries for each run. All objective means use actual local draws with explicit advancing seeds. On budget stop retain the best evaluated parameters. The ordinary run uses the source squared one-hot penalty; the Grover mixer retains the Dicke preparation and literal controlled rotation and uses the unpenalized objective. No hosted execution or historical optimizer trajectory equivalence.

## Change the quantum mixer without hiding the objective

The visible roots express the ordinary penalized phase/mixer and the one-hot
Grover composition. Their reusable state preparation and reflection helpers
remain linked in the parent native module. FlagQuantum constructs coherent
phases, checked inverse regions and gate composition. The classical optimizer
selects six static angles from actual sampled costs.

Try the same nonzero angle list in both visible programs and compare named
probabilities, feasible mass and objective values separately. The source's
literal controlled rotation is retained; do not infer general constraint
preservation solely from the Grover-mixer label. Thirty objective calls are a
budget, not a promise of an optimum. The two original optimizations retain
advancing sample streams and independently sampled final distributions.


## Set up the four-qubit comparison

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.algorithms.search_and_optimization.grover_mixer_qaoa.gm_qaoa.overview.main import (
    print_result,
)
from examples.synthesis.algorithms.search_and_optimization.grover_mixer_qaoa.gm_qaoa.overview.plots import (
    distributions,
    objective_trace,
)
from examples.synthesis.algorithms.search_and_optimization.grover_mixer_qaoa.gm_qaoa.overview.quantum import (
    build_program,
)
from examples.synthesis.algorithms.search_and_optimization.grover_mixer_qaoa.gm_qaoa.overview.workflow import (
    INITIAL,
    cost,
    optimize,
)

directory = Path("artifacts/development/grover-mixer-qaoa")
print("Ordinary costs:", [cost(word, grover=False) for word in range(16)])

import math
from flagquantum import synth
from flagquantum.synth import (
    RX,
    Output,
    QArray,
    QBit,
    allocate,
    hadamard_transform,
    phase,
    qfunc,
)
from examples.synthesis.algorithms.search_and_optimization.grover_mixer_qaoa.gm_qaoa.main import (
    initial_state,
    mixer,
)


@qfunc
def ordinary_ansatz(params: list[float], x: Output[QArray[QBit]]) -> None:
    allocate(4, x)
    hadamard_transform(x)
    for layer in range(3):
        phase(
            (x[0] + x[1] + x[2] + x[3] - 1) ** 2 + x[0] + 2 * x[1] - x[2] + 12 * x[3],
            params[layer],
        )
        for bit in range(4):
            RX(params[layer + 3], x[bit])


@qfunc
def grover_mixer_ansatz(params: list[float], x: Output[QArray[QBit]]) -> None:
    allocate(4, x)
    initial_state(x)
    for layer in range(3):
        phase(x[0] + 2 * x[1] - x[2] + 12 * x[3], params[layer])
        mixer(params[layer + 3], x)


## Define the ordinary mixer

In [ ]:
print("Hadamards prepare a uniform state; RX mixes each decision bit.")


## Use the three-layer source schedule

In [ ]:
print("Initial cost and mixer angles:", INITIAL)


## Synthesize ordinary QAOA

In [ ]:
ordinary_preview = synth.synthesize(
    ordinary_ansatz,
    classical_inputs={"params": INITIAL},
    max_qubits=4,
    max_workspace=0,
    max_operations=2048,
    memory_limit_bytes=1024**2,
)
print(ordinary_preview.report["resources"])


## Optimize the ordinary finite-shot objective

In [ ]:
ordinary = optimize(grover=False)
print_result("Ordinary", ordinary)
display(
    Image(
        filename=str(
            objective_trace(
                ordinary, "Ordinary QAOA objective", directory / "ordinary-trace.png"
            )
        )
    )
)


## Check the actual final sample count

In [ ]:
print("Ordinary observed draws:", sum(row.count for row in ordinary.observations))


## Inspect ordinary samples

In [ ]:
for row in ordinary.observations:
    print(f"{row.word:04b}", row.count, row.cost, row.one_hot)


## Use the one-hot Dicke initial state

In [ ]:
print("One-hot objective costs:", [cost(word, grover=True) for word in (1, 2, 4, 8)])


## Use the Grover mixer

In [ ]:
print("Undo Dicke preparation, apply the source controlled phase, restore preparation.")


## Reuse the three-layer schedule

In [ ]:
print("Grover-mixer initial angles:", INITIAL)


## Synthesize the Grover-mixer circuit

In [ ]:
grover_preview = synth.synthesize(
    grover_mixer_ansatz,
    classical_inputs={"params": INITIAL},
    max_qubits=4,
    max_workspace=0,
    max_operations=2048,
    memory_limit_bytes=1024**2,
)
print(grover_preview.report["resources"])


## Optimize the Grover finite-shot objective

In [ ]:
grover = optimize(grover=True)
print_result("Grover", grover)
display(
    Image(
        filename=str(
            objective_trace(
                grover, "Grover-mixer objective", directory / "grover-trace.png"
            )
        )
    )
)


## Check the independent Grover final draw

In [ ]:
print("Grover observed draws:", sum(row.count for row in grover.observations))


## Inspect Grover samples

In [ ]:
for row in grover.observations:
    print(f"{row.word:04b}", row.count, row.cost, row.one_hot)


## Compare both actual distributions

In [ ]:
display(
    Image(
        filename=str(
            distributions(ordinary, grover, directory / "sample-comparison.png")
        )
    )
)
